The goal is to create a RL agent that is capable of finding the pid gains for a quadcopter control system

In [2]:
!pip install gym pybullet numpy stable-baselines3

In [ ]:
!pip install gym

In [5]:
!pip install 'shimmy>=2.0'

In [25]:
import gym
import numpy as np
import pybullet as p
import pybullet_data
from gym import spaces

class QuadcopterPIDEnv(gym.Env):
    def __init__(self):
        super(QuadcopterPIDEnv, self).__init__()

        # Initialize PyBullet
        self.client = p.connect(p.DIRECT)  
        p.setAdditionalSearchPath(pybullet_data.getDataPath())
        p.setGravity(0, 0, -9.81)

        # Load Quadcopter Model (Replace with actual URDF later)
        self.plane = p.loadURDF("/Users/ortol/Desktop/Falco/Local_Control/bullet3-master/data/plane.urdf")
        self.drone = p.loadURDF("/Users/ortol/Desktop/Falco/Local_Control/bullet3-master/data/Quadrotor/quadrotor.urdf", basePosition=[0, 0, 1], useFixedBase=False)

        # PID gains for altitude & attitude control
        self.Kp_alt = 1.0  
        self.Ki_alt = 0.1  
        self.Kd_alt = 0.01  

        self.Kp_att = np.array([1.0, 1.0, 1.0])  # Roll, Pitch, Yaw
        self.Ki_att = np.array([0.1, 0.1, 0.1])
        self.Kd_att = np.array([0.01, 0.01, 0.01])

        # Define action space (PID tuning for altitude + attitude)
        self.action_space = spaces.Box(low=-0.1, high=0.1, shape=(18,), dtype=np.float32)

        # Define observation space (Position, velocity, attitude, angular velocity)
        self.observation_space = spaces.Box(low=-10, high=10, shape=(12,), dtype=np.float32)

        # Target Position and Orientation (Hover at 2m with no rotation)
        self.target_position = np.array([0, 0, 2])
        self.target_attitude = np.array([0, 0, 0])  # Roll, Pitch, Yaw
        self.target_quaternion = p.getQuaternionFromEuler(self.target_attitude)

        self.prev_error_alt = 0
        self.integral_error_alt = 0
        self.prev_error_att = np.zeros(3)
        self.integral_error_orn = np.zeros(3)
        self.integral_error_pos = np.zeros(3)
        self.prev_error_pos = np.zeros(3)
        self.prev_error_orn = np.zeros(3)
        self.Ts = 0.01  
        self.g = 9.81
        self.m = 3.7 # Mass of the quadcopter
        self.max_thrust = 20 # To be tuned
        self.min_thrust = 0
        self.min_torque = -5
        self.max_torque = 5
        self.timesteps = 0
        self.Done = False
        

    def reset(self, seed=None, options=None):
        """Reset the environment to a random initial state."""
        super().reset(seed=seed)  # ✅ Correct seeding for Gym >= 0.26

        random_pos = np.random.uniform([-0.5, -0.5, 1], [0.5, 0.5, 2])
        random_orn = p.getQuaternionFromEuler(np.random.uniform([-0.1, -0.1, -0.1], [0.1, 0.1, 0.1]))

        p.resetBasePositionAndOrientation(self.drone, random_pos, random_orn)
        p.resetBaseVelocity(self.drone, [0, 0, 0], [0, 0, 0])

        self.state = np.hstack((random_pos, p.getEulerFromQuaternion(random_orn), np.zeros(6)))

        if self.state.shape[0] == 0:  # Check if empty
            raise ValueError("Reset function is returning an empty observation!")

        print("Reset Observation:", self.state)  # Debugging

        return self.state, {}

    def _get_state(self):
        """Get quadcopter state: position, velocity, attitude, angular velocity"""
        pos, orn = p.getBasePositionAndOrientation(self.quadcopter)
        vel, ang_vel = p.getBaseVelocity(self.quadcopter)
        roll, pitch, yaw = p.getEulerFromQuaternion(orn)

        return np.array(pos + vel + (roll, pitch, yaw) + ang_vel)
    
    def step(self, action):
        """Perform one step in the simulation based on PID-tuned gains for (x, y, z) and (roll, pitch, yaw)."""
        action = np.array(action).reshape(3, 6)  # Reshape action into PID gains

        Kp, Ki, Kd = action[0][:], action[1][:], action[2][:]  # Extract gains

        # Get drone state
        pos, orn = p.getBasePositionAndOrientation(self.drone)
        euler = np.array(p.getEulerFromQuaternion(orn))
        vel, ang_vel = p.getBaseVelocity(self.drone)

        # Compute position errors
        curr_error_pos = self.target_position - np.array(pos)
        curr_error_orn = np.array(self.target_attitude) - euler

        # Compute integral and derivative errors
        self.integral_error_pos += curr_error_pos * self.Ts
        self.integral_error_pos = np.clip(self.integral_error_pos, -1, 1)

        self.integral_error_orn += curr_error_orn * self.Ts
        self.integral_error_orn = np.clip(self.integral_error_orn, -1, 1)

        derivative_error_pos = (curr_error_pos - self.prev_error_pos) / float(self.Ts)
        #derivative_error_pos = np.clip(derivative_error_pos, -5, 5)

        derivative_error_orn = (curr_error_orn - self.prev_error_orn) / float(self.Ts)
        #derivative_error_orn = np.clip(derivative_error_orn, -5, 5)

        # **Compute PID control for X, Y, Z position**
        thrust_x = (Kp[0] * curr_error_pos[0] +
                    Ki[0] * self.integral_error_pos[0] +
                    Kd[0] * derivative_error_pos[0])
        
        thrust_y = (Kp[1] * curr_error_pos[1] +
                    Ki[1] * self.integral_error_pos[1] +
                    Kd[1] * derivative_error_pos[1])
        
        thrust_z = (Kp[2] * curr_error_pos[2] +
                    Ki[2] * self.integral_error_pos[2] +
                    Kd[2] * derivative_error_pos[2])

        # Compute PID control for roll, pitch, yaw
        tau_roll = (Kp[3] * curr_error_orn[0] +
                    Ki[3] * self.integral_error_orn[0] +
                    Kd[3] * derivative_error_orn[0])
        
        tau_pitch = (Kp[4] * curr_error_orn[1] +
                    Ki[4] * self.integral_error_orn[1] +
                    Kd[4] * derivative_error_orn[1])
        
        tau_yaw = (Kp[5] * curr_error_orn[2] +
                Ki[5] * self.integral_error_orn[2] +
                Kd[5] * derivative_error_orn[2])

        # Clip forces and torques
        thrust_x = np.clip(thrust_x, -self.max_thrust, self.max_thrust)
        thrust_y = np.clip(thrust_y, -self.max_thrust, self.max_thrust)
        thrust_z = np.clip(thrust_z + self.m * self.g, self.min_thrust, self.max_thrust)

        tau_roll = np.clip(tau_roll, self.min_torque, self.max_torque)
        tau_pitch = np.clip(tau_pitch, self.min_torque, self.max_torque)
        tau_yaw = np.clip(tau_yaw, self.min_torque, self.max_torque)

        # Apply forces and torques
        p.applyExternalForce(self.drone, -1, [thrust_x, thrust_y, thrust_z], [0, 0, 0], p.WORLD_FRAME)
        p.applyExternalTorque(self.drone, -1, [tau_roll, tau_pitch, tau_yaw], p.WORLD_FRAME)

        # Update simulation
        p.stepSimulation()
        self.timesteps += 1

        # Normalize observation
        pos = np.array(pos)  # Convert tuple to NumPy array
        vel = np.array(vel)
        ang_vel = np.array(ang_vel)
        euler_norm = np.array(euler)

        pos_norm = np.clip(pos / 10, -1, 1)  # Normalize position
        vel_norm = np.clip(vel / 5, -1, 1)  # Normalize velocity
        ang_vel_norm = np.clip(ang_vel / 5, -1, 1)  # Normalize angular velocity
        euler_norm = np.clip(euler / np.pi, -1, 1)

        self.state = np.hstack((pos_norm, euler_norm, vel_norm, ang_vel_norm))

        # Reward shaping
        reward =  np.linalg.norm(curr_error_pos)  # Penalize position error
        reward += 0.3 * np.linalg.norm(curr_error_orn)  # Penalize orientation error
        reward += 0.05 * np.linalg.norm(vel)  # Encourage smooth flight
        reward += 0.05 * np.linalg.norm(ang_vel)  # Reduce excessive rotations

        if self.timesteps == 100000:
            self.Done = True

        """
        # Ground collision check
        if pos[2] < 0.1:
            done = True
            reward -= 50"""

        # Update errors for next step
        self.prev_error_pos = curr_error_pos
        self.prev_error_orn = curr_error_orn

        return self.state, reward, self.Done, False, {}


    def close(self):
        p.disconnect()


In [26]:
import gym
from stable_baselines3 import TD3
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3 import PPO

# Create the quadcopter environment
#env = make_vec_env(QuadcopterPIDEnv, n_envs=1)

# Define the TD3 model
"""model = TD3(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=1e-3,       # Fixed learning rate
    buffer_size=1000000,      # Replay buffer size
    batch_size=256,           # Larger batch size for stability
    tau=0.005,                # Soft update coefficient
    gamma=0.99,               # Discount factor
    train_freq=(1, "episode"), # Train after each episode
    gradient_steps=1,         # Number of gradient steps
    policy_delay=2,           # Delayed policy updates
    action_noise=None         # Add noise for exploration if needed
)"""

env = QuadcopterPIDEnv()
model = PPO("MlpPolicy", env, verbose=1, learning_rate=1e-5)
model.learn(total_timesteps=100000)
model.save("drone_pid_ppo")

# Print reward
print("Reward:", model.policy)

# Save the trained model
#model.save("td3_pid_tuning")

print("Training complete!")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Reset Observation: [-0.46010385 -0.44647967  1.52194201  0.01001033  0.04584093 -0.03109347
  0.          0.          0.          0.          0.          0.        ]
-----------------------------
| time/              |      |
|    fps             | 930  |
|    iterations      | 1    |
|    time_elapsed    | 2    |
|    total_timesteps | 2048 |
-----------------------------
-------------------------------------------
| time/                   |               |
|    fps                  | 870           |
|    iterations           | 2             |
|    time_elapsed         | 4             |
|    total_timesteps      | 4096          |
| train/                  |               |
|    approx_kl            | 8.7251625e-05 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -25.5         |
|    explained_variance   | 5.84e-06      |
| 

KeyboardInterrupt: 

In [ ]:
import gym
from stable_baselines3 import TD3
from quad_pid_env import QuadcopterPIDEnv

# Load trained RL model
env = QuadcopterPIDEnv()
model = TD3.load("td3_pid_tuning")

# Test the trained agent
obs = env.reset()
for _ in range(1000):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, _ = env.step(action)
    print(f"PID Gains: {action}, Reward: {reward}")

    if done:
        obs = env.reset()

env.close()

Version = 4.1 Metal - 89.3
Vendor = Apple
Renderer = Apple M3
b3Printf: Selected demo: Physics Server
startThreads creating 1 threads.
starting thread 0
started thread 0 
MotionThreadFunc thread started


/Users/ortol/miniforge3/lib/python3.12/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation done: Robot fell.
Simulation don

: 